# AuraScribble Auto-Retrain — Kaggle Edition (V8)

Triggered automatically by GitHub Actions when ≥500 new Firebase corrections accumulate.

**Flow:**
1. Read Firebase service account from Kaggle Secrets
2. Read base bundle (code + datasets) from attached Kaggle Dataset
3. Download latest checkpoint from Firebase Storage (`models/checkpoint_best.pt`)
4. Download all new corrections from `training_data/new/`
5. Build manifests + train V8 (CTC hybrid, 20 epochs)
6. Upload new ONNX + checkpoint to Firebase
7. Archive processed corrections to `training_data/processed/`

**No human interaction required.** The notebook is idempotent and safe to re-run.

In [ ]:
import os, sys, json, shutil
from pathlib import Path

# === Kaggle paths ===
BUNDLE_BASE = Path('/kaggle/input/aurascribble-bundle')   # attached dataset
WORK_DIR    = '/kaggle/working/handwriting-model'
OUT_DIR     = '/kaggle/working/handwriting-model/output'

# The user's zip might have packed the contents under a top-level
# `colab_bundle/` folder. Auto-detect either layout.
if (BUNDLE_BASE / 'src').is_dir():
    BUNDLE_DIR = BUNDLE_BASE
elif (BUNDLE_BASE / 'colab_bundle' / 'src').is_dir():
    BUNDLE_DIR = BUNDLE_BASE / 'colab_bundle'
else:
    # Last-ditch: pick the first child dir that has src/
    candidates = [d for d in BUNDLE_BASE.iterdir() if d.is_dir() and (d / 'src').is_dir()]
    if not candidates:
        raise RuntimeError(
            f'Could not find src/ under {BUNDLE_BASE}. Contents: '
            f'{list(BUNDLE_BASE.iterdir())[:20]}'
        )
    BUNDLE_DIR = candidates[0]
print(f'Bundle root: {BUNDLE_DIR}')

# Mirror the bundle into a writable working directory
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

# Copy code + processed data (~2.2GB but fast on Kaggle SSD)
for sub in ('src', 'configs', 'scripts', 'data', 'requirements.txt'):
    s = BUNDLE_DIR / sub
    d = Path(WORK_DIR) / sub
    if s.is_dir():
        shutil.copytree(s, d)
    elif s.is_file():
        shutil.copy2(s, d)

os.chdir(WORK_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
print('cwd =', os.getcwd())
print('contents:', sorted(os.listdir('.')))
print('src/ has', len(os.listdir('src')), 'files,', 'configs/ has', len(os.listdir('configs')), 'files')

In [ ]:
# Install deps + Hebrew fonts. Kaggle sometimes hands out P100 GPUs (sm_60) which
# the latest PyTorch doesn't support — pin to a version that still has sm_60 + cu118.
!pip -q install --index-url https://download.pytorch.org/whl/cu118 \
    "torch==2.3.1+cu118" "torchvision==0.18.1+cu118"
!pip -q install firebase-admin google-cloud-storage google-auth PyYAML onnx onnxruntime
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y fonts-noto-core fonts-dejavu fonts-dejavu-core > /dev/null 2>&1
!fc-list :lang=he | head -3

# Sanity: confirm CUDA is functional on this GPU
import torch
print(f'torch={torch.__version__}  cuda_available={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'  capability=sm_{cap[0]}{cap[1]}')
    # Smoke test the GPU
    a = torch.randn(8, 8, device='cuda')
    b = a @ a
    print('  GPU smoke test passed')

In [ ]:
# Firebase service account injected by GitHub Actions before kernel push.
# (Kaggle Secrets can't be attached via the API, so we inline the JSON as base64
# — the kernel is private, only the owner can see it.)
import base64

SA_B64 = "__FIREBASE_SA_B64__"   # replaced by trigger_kaggle_retrain.py
if SA_B64.startswith('__'):
    # fallback for manual runs: try the Kaggle Secrets API
    from kaggle_secrets import UserSecretsClient
    sa_json = UserSecretsClient().get_secret('FIREBASE_SERVICE_ACCOUNT_JSON')
else:
    sa_json = base64.b64decode(SA_B64).decode('utf-8')

cred_path = Path('configs/firebase_service_account.json')
cred_path.parent.mkdir(parents=True, exist_ok=True)
cred_path.write_text(sa_json, encoding='utf-8')
json.loads(sa_json)   # validate
print('✓ Firebase service account loaded')

In [ ]:
# Download previous best checkpoint from Firebase Storage so we can resume.
# We also validate that the checkpoint matches the current model architecture —
# if not, we skip resume rather than confuse the trainer with a broken state_dict.
import firebase_admin
from firebase_admin import credentials, storage

if not firebase_admin._apps:
    firebase_admin.initialize_app(
        credentials.Certificate(str(cred_path)),
        {'storageBucket': 'aurascribblr.firebasestorage.app'},
    )
bucket = storage.bucket()

Path('models').mkdir(exist_ok=True)
blob = bucket.blob('models/checkpoint_best.pt')
RESUME = False
if blob.exists():
    blob.download_to_filename('models/checkpoint_best.pt')
    sz = os.path.getsize('models/checkpoint_best.pt') / 1024 / 1024

    # Quick architecture-compatibility check: peek at the state dict keys.
    # Current model_hybrid expects transformer keys (stem.net, encoder.layers,
    # decoder.layers, ctc_proj). Older runs uploaded an LSTM-based hybrid.
    import torch
    try:
        sd = torch.load('models/checkpoint_best.pt', map_location='cpu', weights_only=False)
        # train.py saves under 'model_state'. Older runs / external dumps may
        # use 'model_state_dict' or 'state_dict' — accept all three.
        if isinstance(sd, dict) and 'model_state' in sd:
            sd = sd['model_state']
        elif isinstance(sd, dict) and 'model_state_dict' in sd:
            sd = sd['model_state_dict']
        elif isinstance(sd, dict) and 'state_dict' in sd:
            sd = sd['state_dict']
        sample_keys = list(sd.keys())[:6] if hasattr(sd, 'keys') else []
        is_transformer = any('encoder.layers' in k or 'stem.net' in k for k in sample_keys)
        is_lstm = any('encoder.weight_ih_l' in k for k in sample_keys)
        print(f'Checkpoint: {sz:.1f} MB, sample keys: {sample_keys}')
        if is_transformer:
            print('✓ Transformer-based checkpoint — will resume from it')
            RESUME = True
        elif is_lstm:
            print('⚠ LSTM-based checkpoint from older run — architecture mismatch, skipping resume')
            os.remove('models/checkpoint_best.pt')
        else:
            print('⚠ Unknown checkpoint format — skipping resume to be safe')
            os.remove('models/checkpoint_best.pt')
    except Exception as e:
        print(f'⚠ Failed to inspect checkpoint ({e}) — skipping resume')
        try:
            os.remove('models/checkpoint_best.pt')
        except OSError:
            pass
else:
    print('⚠ No remote checkpoint yet — training will start from scratch')

# Strip the resumed best_val_cer from the checkpoint. The val set composition
# changes between runs (we keep adding MathWriting samples), so carrying over
# the old best makes every new epoch look "worse" and nothing ever gets saved.
# train.py in the bundle reads `if "val_cer" in checkpoint:` — by deleting that
# key we make it default to inf and let the first new epoch set the baseline.
# This avoids needing to ship a new train.py to the Kaggle Dataset.
if RESUME:
    full = torch.load('models/checkpoint_best.pt', map_location='cpu', weights_only=False)
    if isinstance(full, dict) and 'val_cer' in full:
        old_best = full.pop('val_cer')
        torch.save(full, 'models/checkpoint_best.pt')
        print(f'Stripped val_cer={old_best:.4f} from checkpoint so new epochs can set a fresh baseline.')


In [ ]:
# Download new user corrections from Firebase Storage
import sys
sys.path.insert(0, 'src')
from download_firebase_corrections import download_corrections

n_corrections = download_corrections(
    local_dir='data/raw/training_data/new',
    credentials_path=str(cred_path),
    include_processed=False,
)
print(f'Downloaded {n_corrections} new correction JSON files')

In [ ]:
# Download CROHME 2023 (real handwritten math InkML) into data/raw/
# Adds ~11K real math samples on top of MathWriting's 250K synthetic-leaning
# pool. prepare_raw will auto-discover them via rglob("*.inkml") below.
# Cached across runs: skip if InkML files already present.
import os, subprocess
from pathlib import Path

CROHME23_DIR = Path('data/raw/crohme_2023')
existing_inkml = sum(1 for _ in CROHME23_DIR.rglob('*.inkml')) if CROHME23_DIR.exists() else 0

if existing_inkml >= 1000:
    print(f'CROHME 2023 already extracted: {existing_inkml} InkML files — skipping download.')
else:
    CROHME23_DIR.mkdir(parents=True, exist_ok=True)
    print('Downloading CROHME 2023 (~1.8 GB) from Zenodo...')
    subprocess.run([
        'wget', '-q', '--show-progress',
        'https://zenodo.org/records/8428035/files/CROHME23.zip?download=1',
        '-O', '/tmp/crohme23.zip',
    ], check=True)
    print('Extracting...')
    subprocess.run(['unzip', '-q', '-o', '/tmp/crohme23.zip', '-d', str(CROHME23_DIR)], check=True)
    try:
        os.remove('/tmp/crohme23.zip')
    except OSError:
        pass
    n = sum(1 for _ in CROHME23_DIR.rglob('*.inkml'))
    print(f'CROHME 2023: extracted {n} InkML files into {CROHME23_DIR}')


In [ ]:
# Fold Firebase corrections into all_real.jsonl WITHOUT wiping the 270K baseline
# (IAM-OnDB + CROHME + MathWriting + hebrew_source — all baked into the bundle).
# prepare_raw with append_processed=True reads the existing file first, then adds
# raw discoveries on top and dedups.
from prepare_raw import prepare_raw

n_before = sum(1 for _ in open('data/processed/all_real.jsonl', encoding='utf-8'))
print(f'baseline: {n_before} samples in all_real.jsonl')

n = prepare_raw(
    raw_dir='data/raw',
    output='data/processed/all_real.jsonl',
    append_processed=True,    # CRITICAL: merge with existing 270K, don't overwrite
)
print(f'after merging corrections: {n} samples')

In [ ]:
# Generate synthetic Hebrew + mixed (same as V8)
!python src/generate_synthetic_hebrew.py \
  --output data/raw/synthetic_hebrew/hebrew_synthetic.jsonl \
  --sentences configs/hebrew_sentences_full.txt \
  --variants 40

!python src/generate_synthetic_mixed.py \
  --output data/raw/synthetic_mixed/mixed_synthetic.jsonl \
  --per-template 50

In [ ]:
# Merge real + synthetic into data/processed/all.jsonl
out = Path('data/processed/all.jsonl')
out.parent.mkdir(parents=True, exist_ok=True)
sources = [
    'data/processed/all_real.jsonl',
    'data/raw/synthetic_hebrew/hebrew_synthetic.jsonl',
    'data/raw/synthetic_mixed/mixed_synthetic.jsonl',
]
seen = set()
n_total = 0
with out.open('w', encoding='utf-8') as fout:
    for src in sources:
        sp = Path(src)
        if not sp.exists():
            continue
        for line in sp.open(encoding='utf-8'):
            obj = json.loads(line)
            first = tuple(obj['points'][0]) if obj.get('points') else ()
            key = f"{obj.get('mode')}|{obj.get('text')}|{len(obj.get('points', []))}|{first}"
            if key in seen:
                continue
            seen.add(key)
            fout.write(line if line.endswith('\n') else line + '\n')
            n_total += 1
print(f'all.jsonl: {n_total} samples')

In [ ]:
# RAM-safe streaming split (writes script inline, then runs it).
# Kaggle has 16 GB RAM but with CROHME 2023 (164K InkML) on top of MathWriting
# (250K), the in-memory variant still bursts past the limit. The streaming
# variant scans modes once without strokes, then writes outputs line-by-line.
#
# per-mode-target bumped 6000 -> 15000 so MathWriting contributes 15K to train.

import os
os.makedirs('src', exist_ok=True)
_streaming_src = '"""Streaming variant of split_manifest — RAM-friendly for Colab free (12 GB).\n\nThe original split_manifest.py reads the entire manifest into a Python list,\nwhich blows up to 10+ GB RAM for a 1.5 GB file (Python dict/list overhead).\nThis streaming version makes TWO passes over the file:\n\n  Pass 1: scan only `mode` from each line (cheap) and decide which line index\n          goes to train vs val. With balance-modes, also decide which lines\n          get pruned/kept per mode.\n\n  Pass 2: re-open the file, copy lines verbatim to train.jsonl or val.jsonl\n          based on the decisions from Pass 1.\n\nWe never hold stroke data in memory. Peak RAM is bounded by the per-line-index\nmetadata (~ N × small ints) ≈ 30 MB for 300K samples.\n\nSame CLI as src/split_manifest.py, drop-in replacement.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport random\nimport sys\nfrom pathlib import Path\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--source", required=True, type=Path)\n    parser.add_argument("--train-out", required=True, type=Path)\n    parser.add_argument("--val-out", required=True, type=Path)\n    parser.add_argument("--val-ratio", type=float, default=0.1)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--balance-modes", action="store_true")\n    parser.add_argument("--per-mode-target", type=int, default=15000)\n    parser.add_argument("--max-oversample", type=float, default=8.0)\n    parser.add_argument(\n        "--no-stratify",\n        action="store_true",\n        help="Plain random split rather than per-mode stratified split.",\n    )\n    args = parser.parse_args()\n\n    rng = random.Random(args.seed)\n\n    # ===== Pass 1: scan modes only =====\n    print(f"Pass 1: scanning modes from {args.source}...", flush=True)\n    line_modes: list[str] = []\n    with args.source.open("r", encoding="utf-8") as f:\n        for i, line in enumerate(f):\n            if not line.strip():\n                line_modes.append("")\n                continue\n            try:\n                # Cheap: locate "mode" key without full parse when possible.\n                m_start = line.find(\'"mode"\')\n                if m_start >= 0:\n                    colon = line.find(":", m_start)\n                    q1 = line.find(\'"\', colon)\n                    q2 = line.find(\'"\', q1 + 1)\n                    if q1 >= 0 and q2 > q1:\n                        line_modes.append(line[q1 + 1:q2])\n                        continue\n                # Fallback: full JSON parse\n                obj = json.loads(line)\n                line_modes.append(str(obj.get("mode", "")))\n            except Exception:\n                line_modes.append("")\n            if (i + 1) % 50000 == 0:\n                print(f"  scanned {i + 1} lines, current mode={line_modes[-1]}", flush=True)\n\n    total = len(line_modes)\n    print(f"  total lines: {total}", flush=True)\n\n    # ===== Compute per-mode counts and source histogram =====\n    from collections import Counter\n    source_counts = Counter(line_modes)\n    print(f"  source counts: {dict(source_counts)}", flush=True)\n\n    # ===== Pass 1.5: decide train / val membership =====\n    print("Computing split decisions...", flush=True)\n    train_set: set[int] = set()\n    val_set: set[int] = set()\n\n    if args.no_stratify:\n        all_idx = list(range(total))\n        rng.shuffle(all_idx)\n        n_val = int(round(total * args.val_ratio))\n        val_set.update(all_idx[:n_val])\n        train_set.update(all_idx[n_val:])\n    else:\n        # Stratified split: ~val_ratio of each mode goes to val.\n        per_mode_indices: dict[str, list[int]] = {}\n        for i, m in enumerate(line_modes):\n            per_mode_indices.setdefault(m, []).append(i)\n        for mode, idxs in per_mode_indices.items():\n            local_rng = random.Random(args.seed + hash(mode) % 10000)\n            local_rng.shuffle(idxs)\n            n_val_m = int(round(len(idxs) * args.val_ratio))\n            val_set.update(idxs[:n_val_m])\n            train_set.update(idxs[n_val_m:])\n\n    print(f"  pre-balance train={len(train_set)}, val={len(val_set)}", flush=True)\n\n    # ===== Apply balance-modes to train set =====\n    if args.balance_modes:\n        train_by_mode: dict[str, list[int]] = {}\n        for i in train_set:\n            train_by_mode.setdefault(line_modes[i], []).append(i)\n        new_train: set[int] = set()\n        for mode, idxs in train_by_mode.items():\n            count = len(idxs)\n            cap = min(args.per_mode_target, int(count * args.max_oversample))\n            if count >= cap:\n                # Downsample\n                rng.shuffle(idxs)\n                new_train.update(idxs[:cap])\n            else:\n                # Keep all + oversample by duplicating indices.\n                rng.shuffle(idxs)\n                while len(new_train) < cap and idxs:\n                    # We only add unique indices; for true oversampling we\'d\n                    # need a separate write step. Streaming split keeps just\n                    # the unique survivors; if you need exact oversampling,\n                    # use the non-streaming variant.\n                    new_train.update(idxs[:cap])\n                    break\n        train_set = new_train\n\n    # ===== Compute per-mode summary =====\n    train_counts = Counter(line_modes[i] for i in train_set)\n    val_counts = Counter(line_modes[i] for i in val_set)\n    print(f"  source ({total}): "\n          + ", ".join(f"{k}={v}" for k, v in sorted(source_counts.items())),\n          flush=True)\n    print(f"  train ({len(train_set)}): "\n          + ", ".join(f"{k}={v}" for k, v in sorted(train_counts.items())),\n          flush=True)\n    print(f"  val ({len(val_set)}): "\n          + ", ".join(f"{k}={v}" for k, v in sorted(val_counts.items())),\n          flush=True)\n\n    # ===== Pass 2: stream-write =====\n    print(f"Pass 2: streaming output to {args.train_out} / {args.val_out}", flush=True)\n    args.train_out.parent.mkdir(parents=True, exist_ok=True)\n    args.val_out.parent.mkdir(parents=True, exist_ok=True)\n    n_train = n_val = 0\n    with args.source.open("r", encoding="utf-8") as fin, \\\n            args.train_out.open("w", encoding="utf-8") as ftrain, \\\n            args.val_out.open("w", encoding="utf-8") as fval:\n        for i, line in enumerate(fin):\n            if i in train_set:\n                ftrain.write(line if line.endswith("\\n") else line + "\\n")\n                n_train += 1\n            elif i in val_set:\n                fval.write(line if line.endswith("\\n") else line + "\\n")\n                n_val += 1\n            if (i + 1) % 50000 == 0:\n                print(f"  written {i + 1}/{total} lines (train={n_train}, val={n_val})", flush=True)\n\n    print(f"Wrote {n_train} train + {n_val} val samples", flush=True)\n    print(f"  train: {args.train_out}", flush=True)\n    print(f"  val:   {args.val_out}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
with open('src/split_manifest_streaming.py', 'w', encoding='utf-8') as f:
    f.write(_streaming_src)
print(f'Wrote src/split_manifest_streaming.py ({len(_streaming_src)} bytes)')

!python -u src/split_manifest_streaming.py   --source data/processed/all.jsonl   --train-out data/processed/train.jsonl   --val-out data/processed/val.jsonl   --val-ratio 0.1   --seed 42   --balance-modes   --per-mode-target 10000

!python -u src/build_curriculum_manifest.py   --train data/processed/train.jsonl   --short-out data/processed/train_short.jsonl   --medium-out data/processed/train_medium.jsonl   --iam-out data/processed/train_iam_long.jsonl   --max-chars-short 32   --medium-min-chars 33   --medium-max-chars 72   --rewrite-train


In [ ]:
# Build the V8 training config (hybrid CTC+AR, full train.jsonl, resume if checkpoint exists)
import yaml, copy

with open('configs/train.yaml', 'r', encoding='utf-8') as f:
    base = yaml.safe_load(f)

cfg = copy.deepcopy(base)
cfg['model_type'] = 'hybrid'
cfg['ctc_loss_weight'] = 0.7
cfg['ar_loss_weight']  = 0.3
cfg['learning_rate'] = 3e-5
cfg['training']['learning_rate'] = 3e-5
cfg['train_manifest']        = 'data/processed/train.jsonl'
cfg['data']['train_manifest'] = 'data/processed/train.jsonl'
cfg['resume_from_checkpoint'] = bool(RESUME)
cfg['model_path']             = 'models/checkpoint_best.pt'
cfg['output_dir']             = 'output'
cfg['epochs']                 = 40
cfg['training']['epochs']     = 40
cfg['correction_loss_weight'] = 5.0
cfg['reset_best_val_cer_on_resume'] = True  # val set composition can change between runs

with open('configs/train_kaggle_v8.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f, allow_unicode=True)
print('Wrote configs/train_kaggle_v8.yaml')
print('  resume:', cfg['resume_from_checkpoint'])
print('  epochs:', cfg['epochs'])

In [ ]:
# Train
!python -u src/train.py --config configs/train_kaggle_v8.yaml

# If training didn't beat the prior best (e.g. val set changed), copy the
# resumed checkpoint so predict.py / export_onnx.py can still find something.
import shutil, os
if not os.path.exists('output/checkpoint_best.pt') and os.path.exists('models/checkpoint_best.pt'):
    shutil.copy2('models/checkpoint_best.pt', 'output/checkpoint_best.pt')
    print('No new best produced — using resumed checkpoint for predict/export.')


In [ ]:
# Predict on full val set + collapse check
!python -u src/predict.py \
  --config configs/train_kaggle_v8.yaml \
  --checkpoint output/checkpoint_best.pt \
  --manifest data/processed/val.jsonl \
  --output output/predictions.jsonl

from collections import Counter
preds = {int(json.loads(l)['id']): json.loads(l).get('prediction', '')
         for l in Path('output/predictions.jsonl').open(encoding='utf-8')}
n_total = len(preds)
n_empty = sum(1 for p in preds.values() if not p)
nonempty_unique = len({p for p in preds.values() if p})
diversity = nonempty_unique / max(1, n_total - n_empty)

print(f'total: {n_total}  empty: {n_empty} ({100*n_empty/n_total:.1f}%)')
print(f'nonempty diversity: {diversity:.2f}')

GO = (diversity >= 0.3) and (n_empty < 0.5 * n_total)
print(('✅ Safe to export' if GO else '🚨 NOT SAFE — collapse detected, abort upload'))
if not GO:
    raise RuntimeError('Quality gate failed — refusing to upload broken model')

In [ ]:
# Eval CER
!python -u src/evaluate.py \
  --manifest data/processed/val.jsonl \
  --predictions output/predictions.jsonl \
  --report output/eval_report.json

report = json.loads(Path('output/eval_report.json').read_text(encoding='utf-8'))
print(f"CER mean: {report.get('cer_mean'):.3f}")
print(f"by mode: {report.get('cer_by_mode')}")

In [ ]:
# Export ONNX
!python -u src/export_onnx.py \
  --config configs/train_kaggle_v8.yaml \
  --checkpoint output/checkpoint_best.pt \
  --trace-time 128 --trace-tokens 128 --smoke-time 128 \
  --summary output/export_summary.json

s = json.loads(Path('output/export_summary.json').read_text(encoding='utf-8'))
assert s.get('status') == 'success', f'Export failed: {s}'
print('✓ ONNX export successful')

In [ ]:
# Upload ONNX + checkpoint + vocab to Firebase Storage
uploads = [
    ('output/model.onnx',                   'models/latest_handwriting.onnx'),
    ('output/checkpoint_best.pt',           'models/checkpoint_best.pt'),
    ('output/vocab.from_checkpoint.txt',    'models/latest_vocab.txt'),
]
for local, remote in uploads:
    if os.path.exists(local):
        bucket.blob(remote).upload_from_filename(local)
        print(f'✓ uploaded {local} → gs://{bucket.name}/{remote}')
    else:
        print(f'⚠ missing local file: {local}')

In [ ]:
# Archive processed corrections (move new → processed) so we don't re-train on them
n_moved = 0
for blob in bucket.list_blobs(prefix='training_data/new/'):
    if blob.name.endswith('/') or not blob.name.endswith('.json'):
        continue
    new_path = blob.name.replace('training_data/new/', 'training_data/processed/')
    bucket.rename_blob(blob, new_path)
    n_moved += 1
print(f'Archived {n_moved} correction files to training_data/processed/')

In [ ]:
# Write a result summary so GitHub Actions can confirm success
result = {
    'status': 'success',
    'cer_mean': report.get('cer_mean'),
    'cer_by_mode': report.get('cer_by_mode'),
    'samples': report.get('samples'),
    'corrections_used': n_corrections,
    'archived': n_moved,
}
Path('/kaggle/working/result.json').write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2, ensure_ascii=False))